# 3. Hypothesis Testing

Hypothesis testing provides a formal framework for making decisions from data. This notebook covers:
- One-sample and two-sample **t-tests**
- **Chi-square** test for independence
- **One-way ANOVA**
- **p-values** interpretation and **effect sizes** (Cohen's d, eta-squared)
- Multiple testing correction (Bonferroni)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

%matplotlib inline
np.random.seed(42)

## 3.1 One-Sample t-Test

Test whether the population mean equals a hypothesized value $\mu_0$:
$H_0: \mu = \mu_0$ vs $H_1: \mu \neq \mu_0$. Test statistic: $t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}$

In [ ]:
scores = np.array([72, 68, 75, 71, 69, 74, 73, 67, 76, 70,
                   78, 66, 73, 71, 77, 69, 74, 72, 75, 80])
mu_0 = 70

t_stat, p_value = stats.ttest_1samp(scores, mu_0)
cohens_d = (scores.mean() - mu_0) / scores.std(ddof=1)

print(f"Sample mean: {scores.mean():.2f}")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value:     {p_value:.4f}")
print(f"Decision at alpha=0.05: {'Reject H0' if p_value < 0.05 else 'Fail to reject H0'}")
print(f"Cohen's d: {cohens_d:.4f} ({'small' if abs(cohens_d) < 0.5 else 'medium' if abs(cohens_d) < 0.8 else 'large'} effect)")

## 3.2 Two-Sample t-Test

Compare the means of two independent groups: $H_0: \mu_1 = \mu_2$ vs $H_1: \mu_1 \neq \mu_2$

In [ ]:
group_a = np.random.normal(loc=75, scale=8, size=30)
group_b = np.random.normal(loc=80, scale=9, size=35)

t_stat, p_value = stats.ttest_ind(group_a, group_b, equal_var=False)
pooled_std = np.sqrt((group_a.var(ddof=1)*(len(group_a)-1) + group_b.var(ddof=1)*(len(group_b)-1)) / (len(group_a)+len(group_b)-2))
d = (group_a.mean() - group_b.mean()) / pooled_std
print(f"Group A mean: {group_a.mean():.2f}, Group B mean: {group_b.mean():.2f}")
print(f"t = {t_stat:.4f}, p = {p_value:.4f}, Cohen's d = {d:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(group_a, bins=10, alpha=0.6, label='Group A (Control)', color='steelblue')
ax.hist(group_b, bins=10, alpha=0.6, label='Group B (Treatment)', color='coral')
ax.axvline(group_a.mean(), color='steelblue', ls='--', lw=2)
ax.axvline(group_b.mean(), color='coral', ls='--', lw=2)
ax.legend()
ax.set_title(f'Two-Sample t-Test (p = {p_value:.4f})')
plt.tight_layout()
plt.show()

## 3.3 Chi-Square Test for Independence

Test whether two categorical variables are independent: $\chi^2 = \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$

In [ ]:
observed = np.array([[50, 30], [20, 40]])
chi2, p_value, dof, expected = stats.chi2_contingency(observed)
n = observed.sum()
cramers_v = np.sqrt(chi2 / (n * (min(observed.shape) - 1)))

print("Observed:")
print(pd.DataFrame(observed, index=['Success', 'Failure'], columns=['Group A', 'Group B']))
print(f"\nChi-square = {chi2:.4f}, df = {dof}, p-value = {p_value:.4f}")
print(f"Cramer's V = {cramers_v:.4f}")

## 3.4 One-Way ANOVA with Bonferroni Correction

Compare means across $k \geq 3$ groups ($H_0: \mu_1 = \mu_2 = \cdots = \mu_k$), then perform pairwise tests with Bonferroni-adjusted $\alpha$.

In [ ]:
from itertools import combinations

method_1 = np.random.normal(70, 10, 25)
method_2 = np.random.normal(75, 10, 25)
method_3 = np.random.normal(80, 10, 25)

F_stat, p_value = stats.f_oneway(method_1, method_2, method_3)
all_data = np.concatenate([method_1, method_2, method_3])
grand_mean = all_data.mean()
ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in [method_1, method_2, method_3])
eta_sq = ss_between / np.sum((all_data - grand_mean)**2)

print(f"F = {F_stat:.4f}, p = {p_value:.4f}")
print(f"Eta-squared = {eta_sq:.4f} ({'small' if eta_sq < 0.06 else 'medium' if eta_sq < 0.14 else 'large'} effect)")

# Bonferroni pairwise comparisons
groups = {'M1': method_1, 'M2': method_2, 'M3': method_3}
alpha_bonf = 0.05 / 3
print(f"\nBonferroni alpha = {alpha_bonf:.4f}")
for (n1, g1), (n2, g2) in combinations(groups.items(), 2):
    t, p = stats.ttest_ind(g1, g2, equal_var=False)
    print(f"  {n1} vs {n2}: t={t:.3f}, p={p:.4f} {'*' if p < alpha_bonf else ''}")

fig, ax = plt.subplots(figsize=(7, 4))
bp = ax.boxplot([method_1, method_2, method_3], labels=['Method 1', 'Method 2', 'Method 3'], patch_artist=True)
for patch, color in zip(bp['boxes'], ['#4C72B0', '#55A868', '#C44E52']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_title(f'One-Way ANOVA (F={F_stat:.2f}, p={p_value:.4f})')
ax.set_ylabel('Score')
plt.tight_layout()
plt.show()

## Key Takeaways

- A **p-value** is the probability of observing the data (or more extreme) under $H_0$ -- it is NOT the probability that $H_0$ is true
- Always report **effect sizes** (Cohen's d, eta-squared, Cramer's V) alongside p-values
- Use **Welch's t-test** (default) unless you have strong evidence of equal variances
- Apply **Bonferroni** or other corrections when performing multiple comparisons
- Statistical significance $\neq$ practical significance